<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN — Colab 生产版

本版本面向正式设计任务，不包含官方测试 demo。它会下载官方 `get_model_params.sh` 当前启用的全部权重，修复新版 Colab 的 PyTorch/NumPy 兼容问题，上传单个 PDB 后运行设计，并调用仓库中的 `extract.py` 提取 `overall_confidence` 最高的唯一序列。

In [ ]:
# 0. 检查运行时
import sys, platform, torch
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("建议在 运行时 → 更改运行时类型 中选择 T4 GPU。")

In [ ]:
# 1. 克隆官方仓库并安装 Colab 兼容依赖
from pathlib import Path
import os, shutil, subprocess, sys

ROOT = Path("/content/LigandMPNN")
RESET_REPOSITORY = True

if RESET_REPOSITORY and ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/dauparas/LigandMPNN.git", str(ROOT)],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "ProDy==2.6.1", "biopython>=1.81",
     "ml-collections==0.1.1", "dm-tree==0.1.8"],
    check=True,
)

print("Repository:", ROOT)
subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--oneline"], check=True)

In [ ]:
# 2. 下载官方当前启用的全部 LigandMPNN/ProteinMPNN 权重
MODEL_DIR = ROOT / "model_params"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["bash", str(ROOT / "get_model_params.sh"), str(MODEL_DIR)],
    check=True,
)

weight_files = sorted(MODEL_DIR.glob("*.pt"))
if not weight_files:
    raise RuntimeError("没有下载到任何权重。")

print(f"Downloaded {len(weight_files)} checkpoint files:")
for path in weight_files:
    if path.stat().st_size < 1024**2:
        raise RuntimeError(f"权重文件疑似不完整：{path}")
    print(f"  {path.name}: {path.stat().st_size / 1024**2:.1f} MiB")

In [ ]:
# 3. 修复新版 PyTorch 与 NumPy/OpenFold 兼容性
import os, re

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

run_py = ROOT / "run.py"
source = run_py.read_text(encoding="utf-8")
source = source.replace(
    "torch.load(checkpoint_path, map_location=device)",
    "torch.load(checkpoint_path, map_location=device, weights_only=False)",
)
source = source.replace(
    "torch.load(args.checkpoint_path_sc, map_location=device)",
    "torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)",
)
run_py.write_text(source, encoding="utf-8")

numpy_aliases = {
    r"\bnp\.int\b": "int",
    r"\bnp\.float\b": "float",
    r"\bnp\.bool\b": "bool",
    r"\bnp\.object\b": "object",
    r"\bnp\.str\b": "str",
    r"\bnp\.complex\b": "complex",
}

changed = []
for py_file in ROOT.rglob("*.py"):
    text = py_file.read_text(encoding="utf-8")
    patched = text
    for pattern, replacement in numpy_aliases.items():
        patched = re.sub(pattern, replacement, patched)
    if patched != text:
        py_file.write_text(patched, encoding="utf-8")
        changed.append(py_file.relative_to(ROOT))

print("Compatibility patch completed.")
for path in changed:
    print(" ", path)

In [ ]:
# 4. 获取 extract.py
EXTRACT_PY = ROOT / "extract.py"
subprocess.run(
    ["wget", "-q",
     "https://raw.githubusercontent.com/18217265596/sx/master/extract.py",
     "-O", str(EXTRACT_PY)],
    check=True,
)
if not EXTRACT_PY.exists() or EXTRACT_PY.stat().st_size == 0:
    raise RuntimeError("extract.py 下载失败。")
print("extract.py:", EXTRACT_PY)

## 正式生产参数

`MODEL_TYPE` 与 checkpoint 必须匹配。常用选择：

- `ligand_mpnn`：`ligandmpnn_v_32_005_25.pt`、`010`、`020`、`030`
- `protein_mpnn`：`proteinmpnn_v_48_002.pt`、`010`、`020`、`030`
- `soluble_mpnn`：`solublempnn_v_48_002.pt`、`010`、`020`、`030`
- `per_residue_label_membrane_mpnn`
- `global_label_membrane_mpnn`

对 RFdiffusion 生成的普通蛋白 binder，通常使用 `protein_mpnn`。含小分子、金属或辅因子上下文时使用 `ligand_mpnn`。

In [ ]:
# 5. 上传一个正式生产用 PDB
from google.colab import files

uploaded = files.upload()
pdb_items = [(name, data) for name, data in uploaded.items()
             if name.lower().endswith(".pdb")]
if len(pdb_items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

name, data = pdb_items[0]
INPUT_DIR = ROOT / "user_inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(name).name
USER_PDB.write_bytes(data)
print("PDB:", USER_PDB)

In [ ]:
# 6. 设置正式生产参数
MODEL_TYPE = "ligand_mpnn"
CHECKPOINT_NAME = "ligandmpnn_v_32_010_25.pt"

CHAINS_TO_DESIGN = "A"
SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1

FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = ""

TOP_N = 20

CHECKPOINT_PATH = MODEL_DIR / CHECKPOINT_NAME
if not CHECKPOINT_PATH.exists():
    available = "\n".join(p.name for p in sorted(MODEL_DIR.glob("*.pt")))
    raise FileNotFoundError(
        f"未找到 {CHECKPOINT_NAME}。可用权重：\n{available}"
    )

if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip():
    raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不应同时使用。")

print("Model:", MODEL_TYPE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Total sequences:", BATCH_SIZE * NUMBER_OF_BATCHES)
print("Top N:", TOP_N)

In [ ]:
# 7. 定义完整日志运行函数
from typing import Sequence

def run_and_show(command: Sequence[str], cwd: Path = ROOT):
    env = os.environ.copy()
    env["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    env["PYTHONUNBUFFERED"] = "1"

    print("Running command:\n")
    print(" ".join(map(str, command)))
    print("\n" + "=" * 90)

    result = subprocess.run(
        list(map(str, command)),
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    print("=" * 90)
    print("Exit status:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError("LigandMPNN 运行失败，完整报错见上方。")
    return result

In [ ]:
# 8. 运行正式生产任务
USER_OUT = ROOT / "outputs" / USER_PDB.stem
shutil.rmtree(USER_OUT, ignore_errors=True)

checkpoint_flags = {
    "ligand_mpnn": "--checkpoint_ligand_mpnn",
    "protein_mpnn": "--checkpoint_protein_mpnn",
    "soluble_mpnn": "--checkpoint_soluble_mpnn",
    "per_residue_label_membrane_mpnn": "--checkpoint_membrane_mpnn",
    "global_label_membrane_mpnn": "--checkpoint_global_label_membrane_mpnn",
}

if MODEL_TYPE not in checkpoint_flags:
    raise ValueError(f"不支持的 MODEL_TYPE：{MODEL_TYPE}")

command = [
    sys.executable, "-u", "run.py",
    "--model_type", MODEL_TYPE,
    checkpoint_flags[MODEL_TYPE], str(CHECKPOINT_PATH),
    "--seed", str(SEED),
    "--pdb_path", str(USER_PDB),
    "--out_folder", str(USER_OUT),
    "--batch_size", str(BATCH_SIZE),
    "--number_of_batches", str(NUMBER_OF_BATCHES),
    "--temperature", str(TEMPERATURE),
    "--parse_atoms_with_zero_occupancy",
    str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),
    "--save_stats", str(SAVE_STATS),
    "--verbose", "1",
]

if CHAINS_TO_DESIGN.strip():
    command.extend(["--chains_to_design", CHAINS_TO_DESIGN.strip()])
if FIXED_RESIDUES.strip():
    command.extend(["--fixed_residues", FIXED_RESIDUES.strip()])
if REDESIGNED_RESIDUES.strip():
    command.extend(["--redesigned_residues", REDESIGNED_RESIDUES.strip()])

run_and_show(command)

In [ ]:
# 9. 合并 FASTA，并调用 extract.py 提取高分唯一序列
seq_dir = USER_OUT / "seqs"
fasta_files = sorted(seq_dir.glob("*.fa"))
if not fasta_files:
    raise FileNotFoundError(f"未找到 FASTA：{seq_dir}")

combined_fasta = USER_OUT / "all_sequences.fa"
with combined_fasta.open("w", encoding="utf-8") as out_handle:
    for fasta in fasta_files:
        text = fasta.read_text(encoding="utf-8")
        out_handle.write(text)
        if not text.endswith("\n"):
            out_handle.write("\n")

top_tsv = USER_OUT / f"top_{TOP_N}_unique_sequences.tsv"
result = subprocess.run(
    [sys.executable, str(EXTRACT_PY),
     "-i", str(combined_fasta), "-n", str(TOP_N)],
    cwd=str(ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("extract.py 执行失败。")

top_tsv.write_text(
    "rank\toverall_confidence\tid\tsequence\n" + result.stdout,
    encoding="utf-8",
)
print("Saved:", top_tsv)

In [ ]:
# 10. 同时生成高分序列 FASTA
top_fasta = USER_OUT / f"top_{TOP_N}_unique_sequences.fa"

lines = [
    line.strip() for line in top_tsv.read_text(encoding="utf-8").splitlines()[1:]
    if line.strip()
]

with top_fasta.open("w", encoding="utf-8") as handle:
    for line in lines:
        rank, score, record_id, sequence = line.split("\t", 3)
        handle.write(
            f">rank={rank}, overall_confidence={score}, id={record_id}\n"
            f"{sequence}\n"
        )

print(top_fasta.read_text(encoding="utf-8"))

In [ ]:
# 11. 打包并下载全部正式结果
archive_path = Path(
    shutil.make_archive(
        f"/content/{USER_PDB.stem}_LigandMPNN_results",
        "zip",
        root_dir=str(USER_OUT),
    )
)
print("Archive:", archive_path)
print("Size:", f"{archive_path.stat().st_size / 1024**2:.2f} MiB")
files.download(str(archive_path))